In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

data = [
    (1, "Amit", "Engineering", 75000),
    (2, "Priya", "Marketing", 60000),
    (3, "Rahul", "Engineering", 82000),
    (4, "Sneha", "HR", 55000),
    (5, "Vikram", "Finance", 70000),
]

columns = ["emp_id", "name", "department", "salary"]

df = spark.createDataFrame(data, columns)
df.show()

In [0]:
from pyspark.sql import functions as F
df.filter(F.col("salary")>50000).show()

In [0]:
df.groupBy("department").agg(F.sum("salary").alias("total_salary")).show()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
windowSpec = Window.partitionBy("department").orderBy(F.col("salary").desc())
highest_salary_each_dept = df.withColumn("rank",F.dense_rank().over(windowSpec)).filter(F.col("rank")==1)
display(highest_salary_each_dept)

In [0]:
display(df)

In [0]:
df.dropDuplicates(['emp_id']).show()

In [0]:
# Demo: repartition() vs coalesce()
print(f"Original partitions: {df.rdd.getNumPartitions()}")

# coalesce: reduces partitions WITHOUT a full shuffle (cheap, but partitions may be uneven)
df_coalesced = df.coalesce(1)
print(f"After coalesce(1): {df_coalesced.rdd.getNumPartitions()}")

# repartition: full shuffle, produces evenly-sized partitions (expensive but balanced)
# Also the only way to INCREASE partition count

from pyspark.sql import Row
big_df = spark.range(10_000_000)  # 1 input file -> few large partitions
print(f"big_df partitions before: {big_df.rdd.getNumPartitions()}")

big_df_repartitioned = big_df.repartition(16)
print(f"big_df partitions after repartition(16): {big_df_repartitioned.rdd.getNumPartitions()}")

# Compare per-partition row counts: coalesce keeps skew, repartition balances
import pyspark.sql.functions as F

def partition_sizes(d, label):
    sizes = (d.withColumn("p", F.spark_partition_id())
               .groupBy("p").count()
               .orderBy("p")
               .collect())
    print(f"{label}: " + ", ".join(f"partition {r['p']} -> {r['count']:,} rows" for r in sizes))

partition_sizes(big_df, "Before")
partition_sizes(big_df.coalesce(3), "After coalesce(3)  (no shuffle)")
partition_sizes(big_df.repartition(3), "After repartition(3) (full shuffle, balanced)")